# SAHAi tutor frontier

Five tutoring conditions on the same 60 held-out problems, scored on two axes:
student outcome and answer non-disclosure.

Four arms completed locally on Apple Silicon. The fifth, a prompted
Qwen2.5-3B with no training, reached 9 minutes per problem once swap filled,
with about 7 hours remaining, so the benchmark moved here.

Local solve rates, for reference:

| arm | solve |
|---|---|
| oracle, solution disclosed | 0.850 |
| prompted-3B, 15 of 60 | 0.467 |
| no tutor | 0.450 |
| sahai-base, adapter off | 0.400 |
| sahai-trained | 0.350 |

Leakage and pedagogy decide whether the trained model holds a frontier position
or is simply dominated, and they only print at the end of a complete run.


In [ ]:
import glob
import os
import subprocess
import sys

# The notebook holds no numbers or flags. Everything defining the experiment
# lives in the dataset, because the notebook and the dataset are separate
# uploads and only one of them is usually re-pushed before a run.
#
# The mount path is discovered rather than assumed. Version 2 of this kernel
# died looking for /kaggle/input/<slug> and finding /kaggle/input/datasets, so
# this walks the tree for a directory that actually contains the package.
def find_root(base="/kaggle/input", max_depth=4):
    for depth in range(max_depth):
        pattern = base + "/*" * (depth + 1)
        for cand in sorted(glob.glob(pattern)):
            if os.path.isdir(os.path.join(cand, "sahai")):
                return cand
    return None

ROOT = find_root()
if ROOT is None:
    print("tree under /kaggle/input:")
    for dirpath, dirnames, filenames in os.walk("/kaggle/input"):
        if dirpath.count(os.sep) > 5:
            dirnames[:] = []
            continue
        print(" ", dirpath, "->", sorted(dirnames)[:12])
    raise SystemExit("no attached dataset contains sahai/")

print("package root:", ROOT)
for p in (ROOT, os.path.join(ROOT, "libs", "sahai-core")):
    if os.path.isdir(p):
        sys.path.insert(0, p)
print("contents:", sorted(os.listdir(ROOT)))
print("scripts:", sorted(os.listdir(os.path.join(ROOT, "scripts", "experiments"))))
print("adapter:", sorted(os.listdir(os.path.join(ROOT, "adapter", "best_model"))))


In [ ]:
import torch

# Fail fast on the accelerator. machine_shape in kernel-metadata.json and the
# CLI --accelerator flag are both accepted and then ignored by the server, so a
# misconfigured run silently lands on a P100 whose compute capability this
# torch build ships no kernels for. Stopping at 30 seconds beats stopping at
# nine GPU-hours.
print("torch", torch.__version__, "| cuda available", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("No CUDA. Session options -> Accelerator -> GPU T4 x2.")
cap = torch.cuda.get_device_capability(0)
print("device:", torch.cuda.get_device_name(0), "| capability", cap)
if cap[0] < 7:
    raise SystemExit(
        f"{torch.cuda.get_device_name(0)} is sm_{cap[0]}{cap[1]}; this stack "
        "needs sm_70 or newer. Set the accelerator to GPU T4 x2 in the browser."
    )


In [ ]:
import importlib

import peft

# peft 0.20.0 is already in the image and is the version this adapter was
# trained with, so it is NOT installed or upgraded here. Version 3 of this
# kernel died because of torchao instead.
#
# peft's LoRA dispatcher calls is_torchao_available(), which RAISES on a
# present-but-too-old torchao rather than returning False. Kaggle ships
# torchao 0.10.0 and peft wants above 0.16. Locally torchao is absent, the
# check returns False, and the dispatcher skips that path entirely. Removing
# it here reproduces the environment the adapter was trained in, which is the
# right direction: the alternative, upgrading torchao, changes a quantisation
# library this project does not use on a platform where the training run was
# never validated.
spec = importlib.util.find_spec("torchao")
if spec is not None:
    import subprocess
    import sys

    import torchao

    print(f"removing torchao {torchao.__version__} (peft needs >0.16 or absent)")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"],
                   check=False)
    importlib.invalidate_caches()

from peft.import_utils import is_torchao_available

print("peft", peft.__version__)
try:
    print("is_torchao_available() ->", is_torchao_available())
except ImportError as exc:
    raise SystemExit(f"torchao still incompatible: {exc}")


In [ ]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"
cmd = [
    sys.executable,
    os.path.join(ROOT, "scripts", "experiments", "tutor_frontier.py"),
    "--adapter", os.path.join(ROOT, "adapter", "best_model"),
    "--out", "/kaggle/working/frontier.json",
    "--device", "auto",
]
print(" ".join(cmd), flush=True)
result = subprocess.run(cmd, cwd=ROOT)
print("frontier exit", result.returncode, flush=True)
if result.returncode != 0:
    raise SystemExit(f"frontier failed with {result.returncode}")


In [ ]:
import json

with open("/kaggle/working/frontier.json") as fh:
    out = json.load(fh)
print(json.dumps(out, indent=1)[:4000])
